In [ ]:
top5=[1, 8, 7, 2, 9]
mapping={}
for id,ele in enumerate(top5):
    mapping[ele]=id+1
print(mapping)

In [ ]:
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import json
import numpy as np

In [ ]:
dataset_path='/kaggle/input/datasets/adithip2000/filtered-dataset-zip/filtered_dataset'
annos_path=os.path.join(dataset_path,'annos')
image_path=os.path.join(dataset_path,'image')

In [ ]:
print(dataset_path)
!ls -F "{dataset_path}"

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

#Dataset Loater works on cpu so tensor created for mask rcnn is on cpu only

In [ ]:
import torch

In [ ]:
class MaskRCNNDataset(Dataset):
    def __init__(self,images_dir,annos_dir,transforms=None):
        self.images_dir=images_dir
        self.annos_dir=annos_dir
        self.transforms=transforms
        self.image_files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self,idx):
        image_name=self.image_files[idx]
        image_path=os.path.join(self.images_dir,image_name)
        annos_name=image_name.replace(".jpg",".json")
        annos_path=os.path.join(self.annos_dir,annos_name)
        
        image=cv2.imread(image_path)
        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        height,width=image.shape[:2]
        image=cv2.resize(image, (512, 512))
        
        scale_x = 512 / width
        scale_y = 512 / height



        json_dict=dict()
        with open(annos_path) as f:
            json_dict=json.load(f)
        boxes=[]
        labels=[]
        masks=[]

        for item in json_dict:
            obj=json_dict[item]
            old_cat=obj['category_id']
            if old_cat not in mapping:
                continue
            new_cat=mapping[old_cat]
            
            xmin, ymin, xmax, ymax = obj["bounding_box"]

            scaled_xmin = xmin * scale_x
            scaled_xmax = xmax * scale_x
            scaled_ymin = ymin * scale_y
            scaled_ymax = ymax * scale_y

            # Ensure bounding box has positive height and width
            if scaled_xmax > scaled_xmin and scaled_ymax > scaled_ymin:
                labels.append(new_cat)
                mask = np.zeros((512,512), dtype=np.uint8)
                for polygon in obj["segmentation"]:
                    polygon = np.array(polygon).reshape(-1,2).astype(np.float32)

                    polygon[:,0] = polygon[:,0] * scale_x
                    polygon[:,1] = polygon[:,1] * scale_y

                    polygon = polygon.astype(np.int32)

                    cv2.fillPoly(mask, [polygon], 1)
                masks.append(mask)
                boxes.append([scaled_xmin, scaled_ymin, scaled_xmax, scaled_ymax])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        masks = torch.tensor(np.array(masks), dtype=torch.uint8)

        image = torch.tensor(image, dtype=torch.float32).permute(2,0,1) / 255.0

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks
        }

        return image, target



In [ ]:
dataset = MaskRCNNDataset(image_path, annos_path)
print(len(dataset))


What Does collate_fn Do?

Default DataLoader tries to stack everything.

But here we need:

images = [img1, img2, img3, img4]
targets = [t1, t2, t3, t4]

NOT one big stacked tensor.

So collate_fn preserves them as lists.

In [ ]:
from torch.utils.data import random_split

train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size
generator = torch.Generator().manual_seed(42)

#subset the train dataset

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=generator
)

# subset = torch.utils.data.Subset(train_dataset, range(20000))
# val_sub=torch.utils.data.Subset(train_dataset,range(20000,25000))
# test_sub=torch.utils.data.Subset(val_dataset,range(0,1000))
# randomly choose subset indices
# indices = torch.randperm(len(dataset), generator=generator)

# train_indices = torch.randperm(len(train_dataset), generator=generator)
# train_idx = train_indices[:20000]
# val_idx = train_indices[20000:25000]
# test_idx = torch.arange(1500)

train_idx=torch.load('/kaggle/input/models/adithip2000/indices/pytorch/default/1/train_idx (1).pt')
val_idx=torch.load('/kaggle/input/models/adithip2000/indices/pytorch/default/1/val_idx (1).pt')
test_idx=torch.load('/kaggle/input/models/adithip2000/indices/pytorch/default/1/test_idx (1).pt')

train_sub = torch.utils.data.Subset(train_dataset, train_idx)
val_sub = torch.utils.data.Subset(train_dataset, val_idx)

test_sub = torch.utils.data.Subset(test_dataset,test_idx )

# torch.save(train_idx, "/kaggle/working/train_idx.pt")
# torch.save(val_idx, "/kaggle/working/val_idx.pt")
# torch.save(test_idx, "/kaggle/working/test_idx.pt")


In [ ]:
# from collections import Counter

# class_counts = Counter()

# for idx in train_sub.indices:
#     _, target = dataset[idx]
#     labels = target["labels"].tolist()
#     class_counts.update(labels)

# print(class_counts)

In [ ]:
# class_weights = {}

# for cls, count in class_counts.items():
#     class_weights[cls] = 1.0 / count

In [ ]:
# image_weights=[]
# for idx in train_sub.indices:
#     _, target = dataset[idx]
#     labels = target["labels"].tolist()
#     weight=sum(class_weights[l] for l in labels)/len(labels)
#     image_weights.append(weight)
# print(len(image_weights))

In [ ]:
image_weights=torch.load('/kaggle/input/models/adithip/image-weights-train/pytorch/default/1/image_weights.pt')

In [ ]:
from torch.utils.data import WeightedRandomSampler

sampler = WeightedRandomSampler(
    weights=image_weights,
    num_samples=len(image_weights),
    replacement=True
)

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
from torch.utils.data import DataLoader
#do not use shuffle=true when trying to use sampler
train_loader = DataLoader(
    train_sub,
    batch_size=2,
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_sub,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_sub,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor



In [ ]:
num_classes=len(top5)+1
model=torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT",min_size=512,max_size=512)



👍 In short
weights="DEFAULT"
→ full COCO pretrained model
weights_backbone="DEFAULT"
→ only backbone pretrained

In [ ]:
#replacing box_boundings and mask part
in_features=model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor=FastRCNNPredictor(in_features,num_classes)

#replacing mask predictors from 80 to 6
in_features_mask=model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor=MaskRCNNPredictor(in_features_mask,256,num_classes)


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device="cpu"
model.to(device)
print(device)

In [ ]:
# model_scratch.rpn.post_nms_top_n_train = 500
model.rpn.post_nms_top_n_train = 2000
model.rpn.post_nms_top_n_test = 1000
#This controls how many region proposals the RPN keeps after NMS during training.

In [ ]:
model.roi_heads.detections_per_img = 300

In [ ]:
model.roi_heads.score_thresh = 0.3

In [ ]:
# # Freeze backbone layers
# for param in model.backbone.parameters():
#     param.requires_grad = False

# print("Backbone frozen. Only ROI heads and new layers will be trained.")

In [ ]:
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,
    momentum=0.9,
    weight_decay=0.0005
)

In [ ]:
# #For full fine tuning- method 1 afte method 3
# for param in model.parameters():
#     param.requires_grad = True
# for g in optimizer.param_groups:
#     g["lr"] = 0.001  # smaller LR for fine-tuning

In [ ]:
def validation(model,val_loader,device):
    print("Under Validation...........")
    model.train() # Set to train mode to get loss dict, but will be inside no_grad()
    val_loss = 0.0 # Initialize val_loss
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets] # Ensure targets are on device
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
    return val_loss/len(val_loader)


In [ ]:
def train_one_epoch(model,train_loader,device):
    print("Training........")
    total_img=0
    model.train()
    epoch_loss=0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        # for k, v in loss_dict.items():
        #     print(k, v.item())

        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        #clipping gradients for exploding issues, 
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        epoch_loss += losses.item()
        total_img+=1
        if(total_img%1000==0):
            print(f"epoch loss {losses.item()} image {total_img}/10000")
    return epoch_loss/len(train_loader)


In [ ]:
# checkpoint=torch.load("/kaggle/input/models/adithip/transfer-epoch-15/pytorch/default/1/transfer_checkpoint_epoch14.pth",map_location=torch.device('cuda'))
checkpoint=torch.load("/kaggle/input/models/adithip2000/best-fine-tuned/pytorch/default/1/transfer_checkpoint_epoch48.pth",map_location=torch.device('cuda'))

model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
def mapPred(model,test_loader,device):
    metric=MeanAveragePrecision()
    model.eval()
    for images,targets in test_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with torch.no_grad():
            preds=model(images)
        metric.update(preds,targets)
    result = metric.compute()
    return result

In [ ]:
img=cv2.imread(os.path.join(image_path,'000900.jpg'))
img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

In [ ]:
image_tensor = torch.tensor(img/255.0, dtype=torch.float32)
image_tensor = image_tensor.permute(2,0,1)
image_tensor=image_tensor.to(device)
model.eval()

with torch.no_grad():
    predictions = model([image_tensor])

In [ ]:
pred = predictions[0]

boxes = pred["boxes"]
scores = pred["scores"]
labels = pred["labels"]
masks = pred["masks"]

In [ ]:
# class_names={7:'shorts',1:'short sleeve top',9:'skirt',8:'trousers',2:'long sleeve top'}
# {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}
class_names=['background','short sleeve top','trousers','shorts','long sleeve top','skirt']

In [ ]:
img_draw = img.copy()

for i in range(len(boxes)):

    if scores[i] > 0.9:   # confidence threshold

        x1, y1, x2, y2 = boxes[i]

        x1 = int(x1.item())
        y1 = int(y1.item())
        x2 = int(x2.item())
        y2 = int(y2.item())

        cv2.rectangle(img_draw, (x1,y1), (x2,y2), (0,255,0), 2)
        label_id = labels[i]
        label_name = class_names[label_id]

        text = f"{label_name} {scores[i]:.2f}"
        text_y = max(y1 - 10, 20)

        cv2.putText(
            img_draw,
            text,
            (x1, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255,0,0),
            1
        )

In [ ]:
cv2.imwrite("prediction.jpg", cv2.cvtColor(img_draw, cv2.COLOR_RGB2BGR))

In [ ]:
plt.imshow(img_draw)

In [ ]:
# num_epochs = 120  # start small for testing, 9 first
# torch.cuda.empty_cache()
# # best_loss=50
# best_loss= 0.40659999768435956 
# #best loss after 24th epoch
# count=0
# patience=3

# for epoch in range(24,num_epochs):
#     total_img=0
#     model.train()
#     epoch_loss = 0
#     print(f"{epoch} running...")
#     train_loss=train_one_epoch(model,train_loader,device)
#     val_loss=validation(model,val_loader,device)
#     print(f"train_loss={train_loss},val_loss={val_loss}")
#     if(val_loss<best_loss):
#         print("Saving model....")
#         best_loss=val_loss
#         count=0
#         torch.save(
#             {
#                 "epoch":epoch,
#                 "model_state_dict":model.state_dict(),
#                 "optimizer_state_dict":optimizer.state_dict(),
#             },f"/kaggle/working/transfer_checkpoint_epoch{epoch+1}.pth"
#         )
#     else:
#         print("Not saving.. under patience")
#         count+=1
#     if count >= patience:
#         print("Early stopping triggered")
#         break





In [ ]:
# #Fine tuning based on Map:
# num_epochs = 51  # start small for testing, 9 first
# torch.cuda.empty_cache()
# # best_loss=50
# # best_loss= 0.40659999768435956 
# best_map=0
# #best loss after 24th epoch
# count=0
# patience=3

# for epoch in range(41,num_epochs):
#     total_img=0
#     model.train()
#     epoch_loss = 0
#     print(f"{epoch} running...")
#     train_loss=train_one_epoch(model,train_loader,device)
#     # val_loss=validation(model,val_loader,device)
#     val_map=mapPred(model,val_loader,device)['map']
#     print(f"train_loss={train_loss},val_map={val_map}")
#     if(val_map>best_map):
#         print("Saving model....")
#         best_map=val_map
#         count=0
#         torch.save(
#             {
#                 "epoch":epoch,
#                 "model_state_dict":model.state_dict(),
#                 "optimizer_state_dict":optimizer.state_dict(),
#             },f"/kaggle/working/transfer_checkpoint_epoch{epoch+1}.pth"
#         )
#     else:
#         print("Not saving.. under patience")
#         count+=1
#     if count >= patience:
#         print("Early stopping triggered")
#         break





scratch: 18 epochs, 13 trained

transfer: 9 epochs, total 14 epochs total

In [ ]:
print("Validation begins")
val_loss=validation(model,test_loader,device)
print(val_loss)



In [ ]:
print("Test Map")
result=mapPred(model,test_loader,device)
print(f"map: {result['map']},map_50: {result['map_50']}")

In [ ]:
print(result)

This optimizer line controls **how your model learns during training**. Let’s break every part clearly so you understand **what each parameter does and why it's used in detection models like Mask R-CNN**.

---

# 1️⃣ The optimizer itself: `SGD`

```python
torch.optim.SGD(...)
```

SGD = **Stochastic Gradient Descent**.

During training:

```
prediction → compute loss → compute gradients → update weights
```

SGD updates the weights like:

[
W_{new} = W_{old} - lr \times gradient
]

So the optimizer controls **how much and how fast weights change**.

For **object detection models**, SGD is often preferred because it is **more stable than Adam** for large networks.

---

# 2️⃣ Which parameters are being trained

```python
[p for p in model.parameters() if p.requires_grad]
```

This means:

```
only train parameters where requires_grad = True
```

In your code earlier you froze the backbone:

```python
for param in model.backbone.parameters():
    param.requires_grad = False
```

So now:

```
Backbone → frozen
ROI heads → trainable
```

This line ensures **only the unfrozen layers are optimized**.

---

# 3️⃣ Learning rate

```python
lr = 1e-4
```

Learning rate controls **how big the weight updates are**.

Example:

```
large lr → faster learning but unstable
small lr → slower learning but stable
```

Typical values:

| Stage                | LR          |
| -------------------- | ----------- |
| Training heads       | 1e-4 – 1e-3 |
| Fine-tuning backbone | 1e-5 – 1e-4 |

Your value:

```
lr = 0.0001
```

is good for **fine-tuning Mask R-CNN heads**.

---

# 4️⃣ Momentum

```python
momentum = 0.9
```

Momentum helps the optimizer **keep moving in useful directions**.

Without momentum:

```
gradient → update
```

With momentum:

```
previous update direction also influences next update
```

Think of it like **rolling a ball downhill**.

Benefits:

```
faster convergence
less noisy updates
```

0.9 is the **standard value used in most detection papers**.

---

# 5️⃣ Weight decay

```python
weight_decay = 0.0005
```

Weight decay is **L2 regularization**.

It penalizes very large weights.

Mathematically the loss becomes:

[
Loss_{total} = Loss_{model} + \lambda \sum W^2
]

This helps prevent:

```
overfitting
very large weights
```

For detection models:

```
0.0005
```

is a very common setting.

---

# 6️⃣ Putting everything together

Your optimizer is doing this:

```
Train only unfrozen layers
Use SGD updates
Learning rate = 0.0001
Momentum = 0.9
Weight decay = 0.0005
```

Which is a **very standard configuration for Mask R-CNN fine-tuning**.

---

# 7️⃣ What happens during one training step

For each batch:

```
1. Forward pass
2. Compute loss
3. Compute gradients (backprop)
4. Optimizer updates weights
```

Your code:

```
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

is using this optimizer to update the model.

---

# 8️⃣ Why SGD is often preferred for detection

Many detection papers use:

```
SGD + momentum
```

instead of Adam because:

```
better generalization
more stable training
```

Mask R-CNN papers use very similar settings.

---

# 9️⃣ When you unfreeze the backbone later

If you later do:

```python
for param in model.backbone.parameters():
    param.requires_grad = True
```

then this same optimizer will **start updating backbone weights too**.

Often you reduce learning rate further:

```
lr = 1e-5
```

for safe fine-tuning.

---

✅ So overall, **your optimizer setup is correct and appropriate for transfer learning with Mask R-CNN**.

---

If you'd like, I can also explain **why your training loop might still train slowly even with frozen backbone**, which happens in Mask R-CNN pipelines and often surprises people.


Please explain this error:

The error AssertionError: All bounding boxes should have positive height and width means that some bounding boxes in your dataset are invalid, likely having zero width or height. This can happen if the xmax is not greater than xmin or ymax is not greater than ymin.

I will fix this by modifying the MaskRCNNDataset class. I'll add a check in the __getitem__ method to ensure that only bounding boxes with positive width and height are included. Invalid boxes, along with their associated labels and masks, will be skipped.



Here's why:

Output Format in eval() vs. train():

When a torchvision.models.detection model is in model.eval() mode, if you pass both images and targets, it is designed to output predictions (e.g., detected boxes, labels, masks) rather than a dictionary of loss components. This is its intended behavior for inference.
When the same model is in model.train() mode, it is designed to output a dictionary of loss components (e.g., loss_classifier, loss_box_reg, loss_mask, loss_objectness) when you pass both images and targets. This is what we need to calculate the validation loss.
Preventing Updates with torch.no_grad():

To ensure that the model's weights are not updated during validation (which is the main purpose of validation – just evaluating performance without training), we wrap the entire validation loop within with torch.no_grad():.
torch.no_grad() disables gradient computation, so even though the model is in train() mode (which typically enables gradients), no gradients are calculated, and thus, optimizer.step() would have no effect on validation. This effectively makes it an evaluation run for loss calculation.
So, in summary, we use model.train() to get the correct loss output format from the torchvision.models.detection model, and torch.no_grad() to ensure no actual training (weight updates) occurs during validation.

In [ ]:
# print("num_classes:", num_classes)

# all_labels = set()
# for images, targets in train_loader:
#     for t in targets:
#         for label in t["labels"]:
#             all_labels.add(label.item())

# print("Unique labels:", sorted(all_labels))

In [ ]:
# from torch.utils.data import DataLoader

# def collate_fn(batch):
#     return tuple(zip(*batch))

# # Make shuffle=True for training
# loader = DataLoader(dataset,
#                     batch_size=4,
#                     shuffle=False,
#                     collate_fn=collate_fn)

What Happens Internally

When you write:

for images, targets in loader:

PyTorch automatically:

Calls dataset[0]

Calls dataset[1]

Calls dataset[2]

Calls dataset[3]

Combines them into a batch

You don’t manually loop over indices.

Dataset → returns (image, target)
DataLoader → groups multiple samples
Training loop → processes batch

Very Important Realization

Using loader here means:

Masks are generated

JSON parsed

Tensors created

Then discarded after loop

Nothing permanent is saved.

1. image,target -> dataset is a object if InstanceMaskCreation class with common functions like __init__, __len__ and __getitem__ common for every iterative object. so you need not have to call image path individually, just call dataset[0] no need to iterate over image_files which is a dir list, same this happens in list,dict etc.

2. the second thing is dataset loader loads or iterates automatically through dataset in batches

3. so images,targets in loader loads batches of images -> and returns processed batch of image and  target tensor. Like dataset[0,1,2,3]-> image[0,1,2,3], target[0,1,2,3]

4. dataset[4,5,6,7]->image[4,5,6,7],target[4,5,6,7]

5.

In [ ]:
# import os

# save_dir = "/kaggle/working/processed_dataset"
# os.makedirs(save_dir, exist_ok=True)

In [ ]:
# count = 0

# for images, targets in loader:

#     for i in range(len(images)):

#         sample = {
#             "image": images[i],
#             "target": targets[i]
#         }

#         torch.save(sample, f"{save_dir}/{count}.pt")
#         count += 1

#         if count % 500 == 0:
#             print(count)

class CachedDataset(torch.utils.data.Dataset):

    def __init__(self, processed_dir):
        self.files = sorted(os.listdir(processed_dir))
        self.processed_dir = processed_dir

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.processed_dir, self.files[idx])
        data = torch.load(path)
        return data["image"], data["target"]

In [ ]:
# count=0
# for images, targets in loader:
#     count+=1
#     if (count%500==0 and count!=0):
#         print(count)



    # print(len(images))          # batch size
    # print(len(targets))

    # print(images[0].shape)
    # print(targets[0]["boxes"].shape)

    # break

In [ ]:
# image, target = dataset[0]

# print(image.shape)              # (3, H, W)
# print(target["boxes"].shape)    # (N, 4)
# print(target["masks"].shape)    # (N, H, W)
# print(target["labels"])